## __Aprendizaje no supervisado__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

__Asunto__: Density-based spatial clustering of applications with noise (DBSCAN)

***

In [ ]:
## Libreria
from ucimlrepo import fetch_ucirepo 
import matplotlib.pyplot as plt
import seaborn as sns
from pandas import DataFrame

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

## Carga de datos

__Dataset:__

Estos datos son referenciados en [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/109/wine) que son los resultados de un análisis químico de vinos cultivados en una misma región de Italia pero derivados de cultivos diferentes. 

<center>
    <img src='https://media.licdn.com/dms/image/v2/D5612AQHwzFJW75X22A/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1674384291032?e=2147483647&v=beta&t=IBguDw6af4l1PyaTlV6Rw1YilLGFbUdn2_y178PYdww' width=800>
</center>

In [ ]:
## Cargar objeto de datos
dataset = fetch_ucirepo(id=109)

## Extraer los features del dataset
data = dataset.data.features 

## Mostrar los primeros 5 registros
display(data.head())

## Escalado de los datos
## Instancia del modelo de escalado
scaler = StandardScaler()

## Ajuste del modelo y transformación de los datos
data_scaled = scaler.fit_transform(data)
print('(shape) data: {}'.format(data_scaled.shape))

## Clase DBScan

Para el algoritmo de DBSCAN se utiliza la clase sklearn.cluster.DBSCAN<br>

```{python}
DBSCAN(eps=0.5,min_samples=5, metric='euclidean')
```

| Parámetros | Descripción |
|------------|-------------|
| eps | radio de la esfera n-dimensional para la cual se buscan los minPoints. |
| minPts | número mínimo de puntos dentro de la región definida por eps para considerar a un punto como crítico (el punto a analizar también se cuenta).|
| metric | métrica para el cálculo de la distancia.<br> `scikit-learn` => 'cityblock', 'cosine', 'euclidean','manhattan'.<br> `scipy.spatial.distance` => 'chebyshev', 'correlation', 'hamming', 'jaccard', 'mahalanobis', 'minkowski', 'seuclidean', 'sqeuclidean'.|

<br>

| Atributos | Descripción |
|-----------|-------------|
| core_sample_indices_ | Indices de los puntos core.|
| components_ | Copia de los puntos core.|
| labels_ | Etiquetas de los puntos (-1 implica outlier/noise (N)).|

<br>

|Funciones | Descripción |
|----------|-------------|
| fit(X) | Entrena el modelo con los parametros asignados.|
| fit_predict(X) | Entrena y devuelve los clusters encontrados con los parametros asignados.|

In [ ]:
## Instancia del modelo
model = DBSCAN(eps=1.7, min_samples=3)

## Ajuste del modelo
model.fit(data_scaled)

In [ ]:
## Mostrar las clases generadas
model.labels_

In [ ]:
# Mostrar los puntos CORES (observar que varios puntos no fueron considerados)
model.core_sample_indices_

#### Visualización en 2D mediante PCA

In [ ]:
## Transformación de los datos usando PCA
pca_model = PCA(n_components=2)
pca_model.fit(data_scaled)
pca_data = pca_model.transform(data_scaled)

## Crear un dataframe con los datos transformados mediante PCA y agregar las etiquetas de los clusteres
pca_data = DataFrame(pca_data, columns=['PC1', 'PC2'])
pca_data['cluster'] = model.labels_
pca_data

In [ ]:
## Graficar los datos por clases. 
N = pca_data['cluster'].nunique()

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_data, 
                x='PC1', y='PC2', 
                hue='cluster',
                palette=sns.color_palette("Paired")[:N])
plt.legend(loc=[1.01, 0.5], title='Cluster')
plt.tight_layout()
plt.show()

## Clase NearestNeighbors

#### Buscando eps

Para calcular las distancias de los vecinos se utiliza la clase sklearn.neighbors.NearestNeighbors

```{python}
NearestNeighbors(n_neighbors=5, metric='minkowski', p=2)
```

| Parámetros | Descripción |
|------------|-------------|
| n_neighbors | Números de vecinos a estimar. |
| metric | Métrica para el cálculo de la distancia. |
| p | Parámetro de la distancia de minkowski (2 es euclideana).|

<br>

| Atributos | Descripción |
|-----------|-------------|
| effective_metric_ | Métrica usada para el cálculo de la distancia. |
| effective_metric_params_ | Parámetros de la métrica usada para el cálculo de la distancia. |
| n_samples_fit_ | Número de puntos del dataset.|

<br>

| Funciones | Descripción |
|-----------|-------------|
| fit(X) | Encontrar los vecinos más cercanos de los datos. |
| kneighbors(X, n_neighbors) | Encontrar los K vecinos más cercano de un punto, retorna tanto los índices como la distancia. |
| radius_neighbors(X, radius) | Encontrar los vecinos de uno o más puntos que se encuentran dentro de un radio determinado. |

In [ ]:
## Numero de vecinos a considerar
num_Neighbors=3

## Instancia del modelo
NN_model = NearestNeighbors(n_neighbors=num_Neighbors)

## Ajuste del modelo
NN_model.fit(data_scaled)

## Buscando los vecinos más cercanos
distances, indices = NN_model.kneighbors(data_scaled) 
print(distances.shape)
distances = distances[:, -1]
distances.sort()
distances


In [ ]:
## Numero de puntos
N = len(distances)

plt.figure(figsize=(13, 5))
plt.plot(distances)
plt.hlines(y=2.2, xmin=0, xmax=N, linestyles='dashed', colors='red')
plt.xlabel('i-ésimo punto')
plt.ylabel('Distancia')
plt.title('Distancia al {}° vecino más cercano de cada punto'.format(num_Neighbors))
plt.tight_layout()
plt.grid()
plt.show()

In [ ]:
## Instancia del modelo
model = DBSCAN(eps=2.2, min_samples=3)

## Ajuste del modelo
model.fit(data_scaled)

## Transformación de los datos usando PCA
pca_model = PCA(n_components=2)
pca_model.fit(data_scaled)
pca_data = pca_model.transform(data_scaled)

## Crear un dataframe con los datos transformados mediante PCA y agregar las etiquetas de los clusteres
pca_data = DataFrame(pca_data, columns=['PC1', 'PC2'])
pca_data['cluster'] = model.labels_

## Graficar los datos por clases. 
N = pca_data['cluster'].nunique()

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_data, 
                x='PC1', y='PC2', 
                hue='cluster',
                palette=sns.color_palette()[:N])
plt.legend(loc=[1.01, 0.5], title='Cluster')
plt.tight_layout()
plt.show()